In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import json

In [ ]:
# Load AIDA dataset in backed mode to avoid loading all data into memory
adata = sc.read_h5ad('AIDA.h5ad', backed='r')
print(adata)

AnnData object with n_obs × n_vars = 1265624 × 35477 backed at 'AIDA.h5ad'
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'cell_number_loaded', 'institute', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'donor_BMI_at_collection', 'tissue_type', 'suspension_derivation_process', 'suspension_enriched_cell_types', 'cell_viability_percentage', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'pMito', 'NODG', 'nUMI', 'Country', 'Annotation_Level1', 'Annotation_Level2', 'Annotation_Level3', 'Annotation_Level4', 'Smoking Status', 'cell_type', 'assay', 'disease

In [ ]:
sg_chinese_donors = adata.obs[
    adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese'
]['donor_id'].unique()

np.random.seed(42)
query_donors = np.random.choice(sg_chinese_donors, size=20, replace=False)

print(f"Query donors: {len(query_donors)}")

Query donors: 20


In [ ]:
import json
import pandas as pd

with open('./output/all_reports.json', 'r') as f:
    all_reports = json.load(f)

# B cell metrics at 0% and 100% SG_Chinese reference (repeat 0)
for entry in all_reports:
    if entry['proportion'] in [0.0, 1.0] and entry['repeat'] == 0:
        report = entry['report']
        if 'B cell' in report:
            print(f"\n=== {entry['proportion']*100:.0f}% SG_Chinese ===")
            print(f"B cell - precision: {report['B cell']['precision']:.3f}, "
                  f"recall: {report['B cell']['recall']:.3f}, "
                  f"f1: {report['B cell']['f1-score']:.3f}, "
                  f"support: {report['B cell']['support']}")


=== 0% SG_Chinese ===
B cell - precision: 0.576, recall: 0.507, f1: 0.539, support: 75.0

=== 100% SG_Chinese ===
B cell - precision: 0.542, recall: 0.347, f1: 0.423, support: 75.0


In [ ]:
import json
import numpy as np

with open('./output/all_reports.json', 'r') as f:
    all_reports = json.load(f)

# B cell precision and recall across all proportions (repeat 0)
for entry in all_reports:
    if entry['repeat'] == 0 and 'B cell' in entry['report']:
        r = entry['report']['B cell']
        print(f"{entry['proportion']*100:.0f}%  precision={r['precision']:.3f}  recall={r['recall']:.3f}")

0%  precision=0.576  recall=0.507
10%  precision=0.673  recall=0.440
20%  precision=0.756  recall=0.453
30%  precision=0.550  recall=0.440
40%  precision=0.553  recall=0.347
50%  precision=0.652  recall=0.400
60%  precision=0.562  recall=0.360
70%  precision=0.585  recall=0.320
80%  precision=0.643  recall=0.360
90%  precision=0.596  recall=0.453
100%  precision=0.542  recall=0.347


In [ ]:
import json
import pandas as pd

with open('./output/all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows = []
for entry in all_reports:
    if 'B cell' in entry['report']:
        r = entry['report']['B cell']
        rows.append({
            'proportion': entry['proportion'],
            'repeat': entry['repeat'],
            'precision': r['precision'],
            'recall': r['recall'],
            'f1': r['f1-score']
        })

df = pd.DataFrame(rows)
summary = df.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print(summary)

           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.595  0.057  0.433  0.037  0.499  0.034
0.1            0.628  0.040  0.410  0.029  0.495  0.025
0.2            0.660  0.072  0.414  0.048  0.505  0.041
0.3            0.611  0.059  0.418  0.031  0.495  0.034
0.4            0.630  0.052  0.423  0.042  0.506  0.043
0.5            0.646  0.039  0.424  0.049  0.510  0.039
0.6            0.609  0.064  0.392  0.029  0.476  0.036
0.7            0.591  0.044  0.399  0.044  0.474  0.037
0.8            0.562  0.037  0.410  0.044  0.472  0.033
0.9            0.567  0.035  0.387  0.034  0.459  0.029
1.0            0.543  0.030  0.384  0.029  0.449  0.025


In [ ]:
query_mask = adata.obs['donor_id'].isin(query_donors)
b_cells = adata.obs[query_mask & (adata.obs['cell_type'] == 'B cell')]
print(b_cells[['Annotation_Level1', 'Annotation_Level2',
               'Annotation_Level3', 'Annotation_Level4']].value_counts().head(20))

Annotation_Level1  Annotation_Level2  Annotation_Level3  Annotation_Level4      
B                  B                  B                  flagged_platelet_sum       32
                                                         B_unknown                  20
                                                         flagged_B_low_exp          13
                                                         flagged_B_TNK_gene          6
                                                         flagged_B_platelet_gene     4
Name: count, dtype: int64


In [ ]:
other_ethnicities = ['Japanese', 'Korean', 'Thai', 'Indian',
                     'Singaporean Malay', 'Singaporean Indian']
other_mask = adata.obs['self_reported_ethnicity'].isin(other_ethnicities)
other_b = adata.obs[other_mask & (adata.obs['cell_type'] == 'B cell')]

print(f"Total B cells in other ethnic groups: {len(other_b)}")
print()
print(other_b['Annotation_Level4'].value_counts().head(20))

Total B cells in other ethnic groups: 8048

Annotation_Level4
B_unknown                  4636
flagged_platelet_sum       2364
flagged_B_platelet_gene     548
flagged_B_TNK_gene          265
flagged_B_low_exp           235
CD4+_T_cm                     0
CD4+_T_cm_IFNhi               0
CD4+_T_cyt                    0
CD4+_T_naive_SOX4hi           0
CD4+_T_unknown                0
CD8+_T_GZMBhi                 0
CD8+_T_GZMKhi                 0
CD8+_T_IFNhi                  0
CD8+_T_naive                  0
CD8+_T_naive_IFNhi            0
CD8+_T_unknown                0
CD14+_Monocyte                0
CD14+_Monocyte_IFNhi          0
CD16+_Monocyte                0
CD16+_Monocyte_C1QAhi         0
Name: count, dtype: int64
